In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset_v2.json
/kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json


In [2]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

In [4]:
import pandas as pd

# Load training data
train_df = pd.read_csv(
    "/kaggle/input/datasets/payyavulamadhuri/ag-file/train.csv",
    header=None,
    names=["label", "title", "description"]
)

# Load test data
test_df = pd.read_csv(
    "/kaggle/input/ag-news-classification-dataset/test.csv",
    header=None,
    names=["label", "title", "description"]
)

# Combine title and description
train_df["text"] = train_df["title"].astype(str) + " " + train_df["description"].astype(str)
test_df["text"] = test_df["title"].astype(str) + " " + test_df["description"].astype(str)

# Convert labels from 1-4 to 0-3
train_df["label"] = train_df["label"] - 1
test_df["label"] = test_df["label"] - 1

# Keep only required columns
train_df = train_df[["text", "label"]]
test_df = test_df[["text", "label"]]

print(train_df.head())
print(train_df.shape)
print(train_df["label"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/ag-news-classification-dataset/test.csv'

In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/22895 [00:00<?, ? examples/s]

Map:   0%|          | 0/5724 [00:00<?, ? examples/s]

In [8]:
columns_to_remove = []

if "text" in train_dataset.column_names:
    columns_to_remove.append("text")

if "__index_level_0__" in train_dataset.column_names:
    columns_to_remove.append("__index_level_0__")

train_dataset = train_dataset.remove_columns(columns_to_remove)
test_dataset = test_dataset.remove_columns(columns_to_remove)

In [9]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [12]:
training_args = TrainingArguments(

    output_dir="./roberta_output",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=100,

    report_to="none"
)

In [13]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.420493,0.394178,0.923305
2,0.243869,0.509789,0.921908
3,0.173928,0.636111,0.930643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2148, training_loss=0.32762454053321144, metrics={'train_runtime': 1081.0379, 'train_samples_per_second': 63.536, 'train_steps_per_second': 1.987, 'total_flos': 4517945709350400.0, 'train_loss': 0.32762454053321144, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate()

print(results)

In [15]:
trainer.save_model("/kaggle/working/fine_tuned_roberta")

tokenizer.save_pretrained("/kaggle/working/fine_tuned_roberta")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/fine_tuned_roberta/tokenizer_config.json',
 '/kaggle/working/fine_tuned_roberta/tokenizer.json')

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

trained_model = trainer.model

trained_model.to(device)

trained_model.eval()

text = "Wow! I absolutely love getting stuck in traffic."

inputs = tokenizer(

    text,

    return_tensors="pt",

    truncation=True,

    padding=True,

    max_length=128
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():

    outputs = trained_model(**inputs)

prediction = outputs.logits.argmax(dim=1).item()

print("Predicted Label:", prediction)

if prediction == 1:

    print("Sarcastic")

else:

    print("Non-Sarcastic")

Predicted Label: 0
Non-Sarcastic
